In [ ]:
from google.colab import files
import os
import re
import unicodedata
from collections import Counter
import csv

In [ ]:
uploaded = files.upload()

In [ ]:
file_map = {
    "HI": "Hindi_male_mono.txt",
    "GU": "Gujarati_male_mono.txt",
    "MR": "Marathi_male_mono.txt"
}

os.makedirs("data/processed", exist_ok=True)
os.makedirs("data/analysis", exist_ok=True)

In [ ]:
hindi_map = {
'अ':'a','आ':'aa','इ':'i','ई':'ii','उ':'u','ऊ':'uu','ऋ':'rq','ए':'ee','ऐ':'ei','ओ':'o','औ':'ou','ऑ':'ax',
'ा':'aa','ि':'i','ी':'ii','ु':'u','ू':'uu','ृ':'rq','े':'ee','ै':'ei','ो':'o','ौ':'ou',
'क':'k a','ख':'kh a','ग':'g a','घ':'gh a','ङ':'ng a',
'च':'c a','छ':'ch a','ज':'j a','झ':'jh a','ञ':'nj a',
'ट':'tx a','ठ':'txh a','ड':'dx a','ढ':'dxh a','ण':'nx a',
'त':'t a','थ':'th a','द':'d a','ध':'dh a','न':'n a',
'प':'p a','फ':'ph a','ब':'b a','भ':'bh a','म':'m a',
'य':'y a','र':'r a','ल':'l a','व':'w a','ळ':'lx a',
'श':'sh a','ष':'sx a','स':'s a','ह':'h a',
'ं':'q','ः':'hq','ँ':'mq','्':'',
'ड़':'dxq a','ढ़':'dxhq a','ज़':'z a','फ़':'f a','ख़':'khq a','ग़':'gq a','क़':'kq a'
}

In [ ]:
gujarati_map = {
'અ':'a','આ':'aa','ઇ':'i','ઈ':'ii','ઉ':'u','ઊ':'uu','એ':'ee','ઐ':'ei','ઓ':'o','ઔ':'ou','ઑ':'ax','ઍ':'ae',
'ા':'aa','િ':'i','ી':'ii','ુ':'u','ૂ':'uu','ે':'ee','ૈ':'ei','ો':'o','ૌ':'ou',
'ક':'k a','ખ':'kh a','ગ':'g a','ઘ':'gh a','ઙ':'ng a',
'ચ':'c a','છ':'ch a','જ':'j a','ઝ':'jh a','ઞ':'nj a',
'ટ':'tx a','ઠ':'txh a','ડ':'dx a','ઢ':'dxh a','ણ':'nx a',
'ત':'t a','થ':'th a','દ':'d a','ધ':'dh a','ન':'n a',
'પ':'p a','ફ':'ph a','બ':'b a','ભ':'bh a','મ':'m a',
'ય':'y a','ર':'r a','લ':'l a','વ':'w a','ળ':'lx a',
'શ':'sh a','ષ':'sx a','સ':'s a','હ':'h a',
'ં':'q','ઃ':'hq','ઁ':'mq','્':''
}

In [ ]:
marathi_map = dict(hindi_map)
marathi_map.update({'ळ':'lx a','ऑ':'ax'})

lang_maps = {
    "HI": hindi_map,
    "GU": gujarati_map,
    "MR": marathi_map
}

In [ ]:
def is_valid_word(word):
    if len(word) < 2:
        return False
    for ch in word:
        cp = ord(ch)
        if not (0x0900 <= cp <= 0x097F or 0x0A80 <= cp <= 0x0AFF or unicodedata.category(ch) in ('Mn','Mc')):
            return False
    return True

In [ ]:
all_words = {}

for lang, filename in file_map.items():
    words = set()

    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) < 2:
                continue

            sentence = unicodedata.normalize("NFC", parts[1])
            sentence = re.sub(r"[^\u0900-\u097F\u0A80-\u0AFF\s]", " ", sentence)

            for word in sentence.split():
                word = word.strip()
                if is_valid_word(word):
                    words.add(word)

    all_words[lang] = sorted(words)

    with open(f"data/processed/unique_words_{lang.lower()}.txt","w",encoding="utf-8") as f:
        for w in all_words[lang]:
            f.write(w+"\n")

    print(lang, len(words))

FileNotFoundError: [Errno 2] No such file or directory: 'Hindi_male_mono.txt'

In [ ]:
def word_to_phonemes(word, char_map):
    phonemes = []
    i = 0
    while i < len(word):
        if i+1 < len(word) and word[i:i+2] in char_map:
            val = char_map[word[i:i+2]]
            if val:
                phonemes.extend(val.split())
            i += 2
        elif word[i] in char_map:
            val = char_map[word[i]]
            if val:
                phonemes.extend(val.split())
            i += 1
        else:
            i += 1
    return phonemes

In [ ]:
phoneme_counter = Counter()
lang_ph_counts = {"HI":Counter(),"GU":Counter(),"MR":Counter()}

for lang in ["HI","GU","MR"]:
    word_file = f"data/processed/unique_words_{lang.lower()}.txt"
    out_file = f"data/processed/g2p_{lang.lower()}.txt"

    with open(word_file,"r",encoding="utf-8") as fin, open(out_file,"w",encoding="utf-8") as fout:
        for line in fin:
            word = unicodedata.normalize("NFC", line.strip())
            if not word:
                continue

            phonemes = word_to_phonemes(word, lang_maps[lang])

            if phonemes:
                fout.write(f"{word}\t{' '.join(phonemes)}\n")

                for ph in phonemes:
                    phoneme_counter[ph]+=1
                    lang_ph_counts[lang][ph]+=1

In [ ]:
total_pairs = 0

with open("data/processed/multilingual_g2p_dataset.txt","w",encoding="utf-8") as fout:
    for lang in ["HI","GU","MR"]:
        with open(f"data/processed/g2p_{lang.lower()}.txt","r",encoding="utf-8") as fin:
            for line in fin:
                parts = line.strip().split("\t")
                if len(parts)==2:
                    fout.write(f"<{lang}> {parts[0]}\t{parts[1]}\n")
                    total_pairs += 1

print(total_pairs)

In [ ]:
with open("data/analysis/phoneme_inventory.txt","w",encoding="utf-8") as f:
    f.write(f"Total phonemes: {len(phoneme_counter)}\n")
    f.write(f"Total tokens: {sum(phoneme_counter.values())}\n\n")

    for ph,count in sorted(phoneme_counter.items(), key=lambda x:-x[1]):
        f.write(f"{ph}\t{count}\n")

In [ ]:
with open("data/analysis/phoneme_frequency.csv","w",newline="",encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["phoneme","total","HI","GU","MR"])

    for ph,count in sorted(phoneme_counter.items(), key=lambda x:-x[1]):
        writer.writerow([
            ph,
            count,
            lang_ph_counts["HI"].get(ph,0),
            lang_ph_counts["GU"].get(ph,0),
            lang_ph_counts["MR"].get(ph,0)
        ])

In [ ]:
files.download("data/processed/multilingual_g2p_dataset.txt")
files.download("data/processed/g2p_hi.txt")
files.download("data/processed/g2p_gu.txt")
files.download("data/processed/g2p_mr.txt")
files.download("data/analysis/phoneme_inventory.txt")
files.download("data/analysis/phoneme_frequency.csv")